# HRV input validation and explanation of eligibility losses

This notebook displays development-only **aggregate diagnostics** and reruns **synthetic-only** package validation. The read-only count walkthrough loads private development window/EMA-key metadata internally, but displays only aggregates and synthetic examples. It never loads final-test records or outcomes. No real HRV, PSS targets, imputation, spectral/nonlinear features or models are calculated.

Frozen policies, five-minute windows, one-minute steps, original EMA cutoffs and both interval-boundary hypotheses remain unchanged. Quality diagnostics are not prediction features.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/garmin-matplotlib")
REPORT = ROOT / "reports/hrv_input_validation/d0e494f0d2cbeb42"
summary = json.loads((REPORT / "summary.json").read_text())
assert not summary["real_hrv_calculated"] and not summary["final_test_processed"]
assert summary["all_development_ema_accounted"]
policies = json.loads((ROOT / "config/ibi_eligibility.json").read_text())
frozen = json.loads((ROOT / "reports/ibi_eligibility/974e98729579967b_8ac2c64795c6/run_summary.json").read_text())
assert hashlib.sha256((ROOT / "config/ibi_eligibility.json").read_bytes()).hexdigest() == frozen["inputs"]["policy_sha256"]
print("Corrected aggregate checkpoint; historical reports are preserved, not reused as current windows.")

## Trace the immediate five-minute counts — read-only

This section queries **corrected** saved metadata, not the reported totals. The aggregate report
pointer above now follows Notebook 04's supersession record; historical files remain unchanged.
The corrected diagnostics are `private/hrv_input_validation/diagnostics/55eb9ba094e893b7/`;
the independent pilot census is
`private/ibi_pilot/selection/52eb55d251e1d5e1/ema_availability.parquet`.

Count one observation window per EMA: `n_step_index == 0`. Its stored start/end must equal
`[n_cutoff_epoch - 300, n_cutoff_epoch)`. A logical slice's stop minus start counts recorded IBI
before interval eligibility. Restrict cases/runs to **Screen only** and keep the two hypotheses
separate. `rejection_mask == 0` means the original Screen policy passes.

For the stronger criterion, test `timestamp_span_seconds >= 240` **and** `supported_seconds >= 240`
on the **same run row**, then require the window's Screen pass. Never combine the maximum span of
one run with another run's support, or use total window support as one run's support. Multiple
qualifying runs still count as one EMA. For the pilot table use only `lookback_minutes == 5` and
count `qualifying_windows > 0`, not a sum across eight lookbacks.

The existing `load_indexed_window` hydrates actual intervals; it is unnecessary for this count-only
query. Its provenance guard is reused through `verify_bundle`; manifest links, development metadata,
unique keys and boundaries are checked below. No sensor streams, package features or model fits
are recomputed. Private rows, file-partition names and real clocks are never displayed.

In [ ]:
from garmin_preprocessing.common import verify_bundle
from garmin_inventory.privacy import sha256

DIAGNOSTICS = ROOT / "private/hrv_input_validation/diagnostics/55eb9ba094e893b7"
SELECTION = ROOT / "private/ibi_pilot/selection/52eb55d251e1d5e1"
NORMALIZED = ROOT / "private/preprocessing/normalized/31e51c934bcdeffa"
SCENARIOS = ("interval_end", "interval_start")

# Verify immutable inputs without re-running their historical generating commands.
diagnostic_manifest = verify_bundle(DIAGNOSTICS)
selection_manifest = verify_bundle(SELECTION)
normalized_hash = sha256(NORMALIZED / "manifest.json")
assert diagnostic_manifest["inputs"]["normalized_manifest"] == normalized_hash
assert selection_manifest["inputs"]["normalized_manifest"] == normalized_hash
assert diagnostic_manifest["inputs"]["frozen_policy"] == selection_manifest["inputs"]["policy_sha256"]
assert selection_manifest["inputs"]["assignments_sha256"] == sha256(
    ROOT / "private/frozen_participant_assignments.parquet"
)
supersession = json.loads((ROOT / "reports/window_supersession.json").read_text())
assert supersession["corrected_input_diagnostics"] == str(DIAGNOSTICS.relative_to(ROOT))
assert diagnostic_manifest["inputs"]["eligibility_manifest"] == sha256(
    ROOT / supersession["corrected_eligibility"] / "manifest.json"
)

# Exact source patterns/columns, not private partition filenames or participant rows.
source_columns = {
    "window_index/*.parquet": ["n_ema_id", "n_step_index", "n_window_start_epoch",
        "n_window_end_epoch", "n_fold", "logical_slice_start", "logical_slice_stop"],
    "immediate_cases/*.parquet": ["ema_id", "policy", "scenario", "rejection_mask", "pilot_candidate_runs"],
    "immediate_runs/*.parquet": ["n_ema_id", "c_policy", "c_scenario", "run_id",
        "timestamp_span_seconds", "supported_seconds", "frozen_window_eligible"],
}
ema_columns = ["n_ema_id", "n_partition", "n_fold", "n_cutoff_epoch"]
pilot_columns = ["ema_id", "fold", "scenario", "lookback_minutes", "qualifying_windows", "invalid_cutoff"]
source_display = [{"source_path_or_pattern": str(DIAGNOSTICS.relative_to(ROOT) / pattern),
                   "columns_used": ", ".join(columns)} for pattern, columns in source_columns.items()]
source_display += [
    {"source_path_or_pattern": str(NORMALIZED.relative_to(ROOT) / "ema/development_ema.parquet"),
     "columns_used": ", ".join(ema_columns)},
    {"source_path_or_pattern": str(NORMALIZED.relative_to(ROOT) / "participant_directories.json"),
     "columns_used": "directory, fold (private development partition routing only)"},
    {"source_path_or_pattern": str(SELECTION.relative_to(ROOT) / "ema_availability.parquet"),
     "columns_used": ", ".join(pilot_columns)},
]
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(source_display))

ema_meta = pd.read_parquet(NORMALIZED / "ema/development_ema.parquet", columns=ema_columns)
assert ema_meta.n_partition.eq("development").all() and ema_meta.n_fold.isin(range(5)).all()
assert ema_meta.n_ema_id.notna().all() and ema_meta.n_ema_id.is_unique
people = json.loads((NORMALIZED / "participant_directories.json").read_text())
index_parts, case_parts, run_parts = [], [], []
for person in people:
    private_file = person["directory"] + ".parquet"
    index_part = pd.read_parquet(DIAGNOSTICS / "window_index" / private_file,
        columns=source_columns["window_index/*.parquet"], filters=[("n_step_index", "==", 0)])
    assert index_part.n_fold.eq(person["fold"]).all()
    index_parts.append(index_part)
    case_parts.append(pd.read_parquet(DIAGNOSTICS / "immediate_cases" / private_file,
        columns=source_columns["immediate_cases/*.parquet"], filters=[("policy", "==", "screen")]))
    run_path = DIAGNOSTICS / "immediate_runs" / private_file
    if run_path.exists():  # Some participants have no runs; their EMA rows remain in the index.
        run_parts.append(pd.read_parquet(run_path, columns=source_columns["immediate_runs/*.parquet"],
            filters=[("c_policy", "==", "screen")]))

immediate_index = pd.concat(index_parts, ignore_index=True)
screen_cases = pd.concat(case_parts, ignore_index=True)
screen_runs = (pd.concat(run_parts, ignore_index=True) if run_parts else
               pd.DataFrame(columns=source_columns["immediate_runs/*.parquet"]))
assert immediate_index.n_ema_id.is_unique
assert set(immediate_index.n_ema_id) == set(ema_meta.n_ema_id)
private_clocks = immediate_index.merge(ema_meta, on="n_ema_id", validate="one_to_one")
assert private_clocks.n_fold_x.eq(private_clocks.n_fold_y).all()
assert private_clocks.n_window_end_epoch.eq(private_clocks.n_cutoff_epoch).all()
assert (private_clocks.n_window_end_epoch - private_clocks.n_window_start_epoch).eq(300).all()
assert (immediate_index.logical_slice_stop >= immediate_index.logical_slice_start).all()
recorded_by_ema = immediate_index.set_index("n_ema_id").eval("logical_slice_stop - logical_slice_start")
assert not screen_cases.duplicated(["ema_id", "scenario"]).any()
assert set(screen_cases.scenario) == set(SCENARIOS)
assert not screen_runs.duplicated(["n_ema_id", "c_scenario", "run_id"]).any()
assert set(screen_runs.n_ema_id) <= set(ema_meta.n_ema_id)
assert set(screen_runs.c_scenario) <= set(SCENARIOS)

pilot_immediate = pd.read_parquet(SELECTION / "ema_availability.parquet", columns=pilot_columns,
    filters=[("lookback_minutes", "==", 5)])
assert not pilot_immediate.duplicated(["ema_id", "scenario"]).any()
assert set(pilot_immediate.scenario) == set(SCENARIOS)
assert pilot_immediate.fold.eq(pilot_immediate.ema_id.map(ema_meta.set_index("n_ema_id").n_fold)).all()
assert not pilot_immediate.invalid_cutoff.any()
assert pilot_immediate.qualifying_windows.isin([0, 1]).all()


In [ ]:
counts, reconciliation = [], []
for scenario in SCENARIOS:
    cases = screen_cases.loc[screen_cases.scenario.eq(scenario)].set_index("ema_id")
    availability = pilot_immediate.loc[pilot_immediate.scenario.eq(scenario)].set_index("ema_id")
    assert set(cases.index) == set(availability.index) == set(recorded_by_ema.index)
    cases = cases.reindex(recorded_by_ema.index)
    availability = availability.reindex(recorded_by_ema.index)
    some = recorded_by_ema.gt(0)
    passed = cases.rejection_mask.eq(0)
    runs = screen_runs.loc[screen_runs.c_scenario.eq(scenario)]
    assert runs.frozen_window_eligible.eq(runs.n_ema_id.map(passed)).all()
    qualifying_ids = set(runs.loc[
        runs.timestamp_span_seconds.ge(240) & runs.supported_seconds.ge(240), "n_ema_id"])
    dual240 = passed & pd.Series(cases.index.isin(qualifying_ids), index=cases.index)
    pilot240 = availability.qualifying_windows.gt(0)
    counts.append({"hypothesis": scenario, "all_ema": len(cases),
        "no_recorded_ibi": int((~some).sum()), "some_recorded_ibi": int(some.sum()),
        "screen_pass": int(passed.sum()), "screen_fail_all_ema": int((~passed).sum()),
        "recorded_but_screen_fail": int((some & ~passed).sum()),
        "screen_plus_same_run_span_and_support_ge240": int(dual240.sum()),
        "pilot_table_ge240": int(pilot240.sum())})
    # Only aggregate discrepancy counts are shown, never the underlying EMA keys.
    reconciliation.append({"hypothesis": scenario,
        "recorded_vs_no_recording_reason_disagreements": int((cases.rejection_mask.mod(2).eq(1) != ~some).sum()),
        "screen_pass_without_recording": int((passed & ~some).sum()),
        "run_vs_saved_candidate_status_disagreements": int((dual240 != cases.pilot_candidate_runs.gt(0)).sum()),
        "run_vs_pilot_ema_status_disagreements": int((dual240 != pilot240).sum())})
immediate_counts = pd.DataFrame(counts)
display(immediate_counts)
display(pd.DataFrame(reconciliation))

# Expected numbers are comparison targets only; they never filter or alter observations.
comparison = []
for row in counts:
    expected = {"all_ema": 3706, "no_recorded_ibi": 2196, "some_recorded_ibi": 1510,
        "screen_pass": 4 if row["hypothesis"] == "interval_end" else 5,
        "screen_plus_same_run_span_and_support_ge240": 1, "pilot_table_ge240": 1}
    for measure, reported in expected.items():
        observed = row[measure]
        comparison.append({"hypothesis": row["hypothesis"], "measure": measure,
            "observed": observed, "reported": reported, "difference": observed - reported,
            "status": "match" if observed == reported else "DISCREPANCY"})
count_verification = pd.DataFrame(comparison)
display(count_verification)
if count_verification.difference.ne(0).any() or any(
    value != 0 for row in reconciliation for key, value in row.items() if key != "hypothesis"
):
    print("DISCREPANCY: retain observed counts and investigate; do not force agreement.")
else:
    print("All reported totals and per-EMA status comparisons agree; each EMA counted once per hypothesis.")


`screen_fail_all_ema` includes windows with no IBI; `recorded_but_screen_fail` is its recorded
subset. The stronger criterion is a subset of Screen passes, not an extra independent group.
Two hypothesis rows refer to the same development EMAs and must not be added. A qualifying
≥240-second run is still not complete five-minute HRV; normal-beat status remains unverified.

### Synthetic cutoff timelines — no real survey clocks

Minute-resolution actual start **09:22 → cutoff 09:21 → [09:16,09:21)**.
Second-resolution actual start **09:22:35 → cutoff 09:22:35 → [09:17:35,09:22:35)**.
The left endpoint is included; the right endpoint is excluded. These are synthetic local-clock
illustrations using a constant illustrative offset. Production UTC is local start minus the row's
local-minus-UTC offset, then the precision-specific buffer. Garmin Unix timestamps get no additional
timezone shift. The buffer does not resolve acquisition-versus-receipt semantics.

In [ ]:
from datetime import datetime, timedelta

synthetic_timelines = []
for start_text, precision in [("09:22", "minute"), ("09:22:35", "second")]:
    start = datetime.strptime(start_text, "%H:%M" if precision == "minute" else "%H:%M:%S")
    cutoff = start - timedelta(seconds=60 if precision == "minute" else 0)
    left = cutoff - timedelta(seconds=300)
    clock_format = "%H:%M" if precision == "minute" else "%H:%M:%S"
    synthetic_timelines.append({"synthetic_start": start_text, "precision": precision,
        "synthetic_cutoff": cutoff.strftime(clock_format),
        "synthetic_window": f"[{left.strftime(clock_format)},{cutoff.strftime(clock_format)})"})
assert synthetic_timelines[0]["synthetic_window"] == "[09:16,09:21)"
assert synthetic_timelines[1]["synthetic_window"] == "[09:17:35,09:22:35)"
display(Markdown("**Synthetic timelines only; no participant dates or clocks.**"))
display(pd.DataFrame(synthetic_timelines))


## Why immediate windows fail

Keep missing recordings separate from recorded but rejected intervals. Reasons overlap; their counts must not be added. A window failing only one check is different from one failing that check together with several others.

In [ ]:
overview = pd.DataFrame(json.loads((REPORT / "overview.json").read_text()))
reasons = pd.DataFrame(json.loads((REPORT / "reasons.json").read_text()))
display(overview.loc[overview.survey_category.eq("All")])
r = reasons.loc[reasons.policy.eq("screen") & reasons.scenario.eq("interval_end") & reasons.survey_category.eq("All")]
display(r[["reason", "overlapping", "only_this_check", "pass_if_only_gate_waived"]])
display(Image(filename=str(REPORT / "rejection_explanation.png")))

Among 3,706 immediate windows, **2,196 have no IBI** and 1,510 contain records. Screen/end retains four; Screen/start retains five. Gap limits and insufficient unbroken runs account for the principal recorded-data failures. A high overall fraction of consistent pairs can still contain many run-breaking mismatches.

The gate-waiver column changes only the final decision bit, without rebuilding support/runs. It is a dependency diagnostic, not authorization to use those windows. It never waives the requirement for recorded IBI.

In [ ]:
variations = pd.DataFrame(json.loads((REPORT / "variations.json").read_text()))
display(variations[["variation", "scenario", "eligible", "eligible_participants"]])
display(Image(filename=str(REPORT / "diagnostic_variations.png")))

These are explicitly labeled diagnostics around frozen Screen, not replacement policies. Disabling the provisional >20% jump flags leaves retention at four/end and five/start. Such a flag is not proof of artifact or ectopy. Broader timing tolerance and shorter required runs increase retention but do not establish physiological validity or acquisition semantics.

In [ ]:
precision = json.loads((REPORT / "timestamp_precision.json").read_text())
pairs = json.loads((REPORT / "pair_timing.json").read_text())
display(pd.DataFrame.from_dict(precision, orient="index", columns=["aggregate_count"]))
display(pd.DataFrame.from_dict(pairs, orient="index", columns=["aggregate_count"]))

Most original timestamps carry six or seven nonzero fractional digits, so whole-second formatting cannot explain the problem universally. Precision is not accuracy. Compressed timestamp spacing and long gaps are consistent with several mechanisms, including batching or receipt timing; only the provider can establish which applies. Clocks are never shifted, rounded or reconstructed to improve retention.

In [ ]:
continuity = pd.DataFrame(json.loads((REPORT / "continuity.json").read_text()))
display(continuity.loc[continuity.policy.eq("screen") & continuity.subset.eq("eligible")])
display(Image(filename=str(REPORT / "continuity_explanation.png")))
display(pd.DataFrame([{k: summary[k] for k in ["window_index_rows", "immediate_windows", "immediate_member_case_rows", "immediate_run_rows"]}]))

## Input contract

Distinguish the 300-second observation window, the union of actual hypothesized interval support, each provisional uninterrupted run, and eligibility for a particular calculation. **Screen's 120-second minimum run does not authorize labeling that run's result five-minute HRV.**

All windows have a private index into immutable records. Immediate windows additionally have materialized member flags and run boundaries. Other approved windows hydrate on demand using the same contract. Rejected records remain present with `run_id=-1`; a gap, rejected interval, missing original row or inconsistent pair breaks adjacency. Boundary-crossing intervals are retained and flagged, not clipped or borrowed.

See `reports/hrv_input_specification.md` for exact fields, formulas, package conditions and limitations.

## Synthetic arithmetic and package adapters

For intervals **800, 900, 1000, 950, 850 ms**, the successive differences are +100, +100, −50, −100 ms. Mean and median are 900 ms. Sample SDNN uses N−1; RMSSD uses the count of allowed original adjacent pairs; pNN50 counts absolute differences **strictly greater** than 50 ms and uses that same pair denominator.

These values below are generated fixtures, not participant observations.

In [ ]:
from garmin_hrv_input.synthetic import independent_reference, validate_all

example = independent_reference([800, 900, 1000, 950, 850])
display(pd.DataFrame([example]))
validated = validate_all(policies["policies"][1])
stored = json.loads((REPORT / "synthetic_validation.json").read_text())
assert validated == stored
assert not validated["real_data_hrv_calculated"]
assert not validated["interpolation"]
display(pd.DataFrame.from_dict(validated["package_evidence"]["versions"], orient="index", columns=["installed_version"]))
display(pd.DataFrame([{k: row[k] for k in ["case", "run", "backend", "status"]} for row in validated["comparisons"]]))
print("Synthetic results exactly match the saved checkpoint.")

NeuroKit2 0.2.13 uses sample SDNN but native pNN50 divides by P+1; the adapter explicitly converts to our P denominator after checking pair identity. HeartPy 1.2.7 uses population SDNN and proportional pNN50; the adapter converts them explicitly. HeartPy exposes no median IBI output.

The HeartPy adapter calls only `analysis.calc_ts_measures`, with externally validated absolute differences and squared differences. It does not call `process_rr`, whose high-level pipeline also attempts breathing and Poincaré calculations. NeuroKit receives original timestamp coordinates, never an invented clock; its other time-domain outputs are not retained.

The next table is a **synthetic negative control**, not an approved way to calculate gappy HRV. It shows that NeuroKit filters differences when some timestamp pairs match its 10 ms rule, but can skip that filtering when none match. Native package behavior is not an eligibility decision.

In [ ]:
display(pd.DataFrame(validated["native_probes"])[[
    "case", "native_missing_detected", "native_10ms_matching_pairs", "effective_package_pairs",
    "native_rmssd_ms", "native_pnn50_percent", "reference_conventions_match"]])
matched = sum(row["status"] == "matched" for row in validated["comparisons"])
print(f"{validated['cases']} synthetic cases: {matched} matched run/backend checks; "
      f"{len(validated['comparisons']) - matched} intentionally blocked.")

## What is ready, and what remains uncertain

Verified: export accounting, rejection overlaps, explicit adjacency, package conventions and synthetic arithmetic. Provisional: the frozen numeric thresholds and >20% jump screen. Provider questions: IBI units, sensor modality, acquisition/receipt clock, interval start/end meaning, batching, missed beats and normal-beat annotations. Neither sample retention nor numerical agreement answers those questions.

The next small development pilot should use at most ten unique nonoverlapping runs, at most one per participant, from frozen Screen-eligible windows with **run span ≥240 seconds and supported duration ≥240 seconds**. Use deterministic chronology, preserve both boundary hypotheses, and do not loosen rules to fill a quota. Only one immediate window meets that restriction under both hypotheses. Longer lookbacks can be reviewed using the same index without treating overlapping views as independent samples.

Mean/median IBI, sample-convention SDNN, RMSSD and pair-denominator pNN50 are numerically ready for such a pilot once its source interpretation is documented. Label actual duration and unverified NN status. NeuroKit pair outputs require end semantics and an exactly compatible pair set; otherwise use the validated explicit-adjacency arithmetic. No real pilot values are calculated in this notebook. Spectral and additional nonlinear features remain deferred.